# 06 · Weather API (Open-Meteo)

- **Source:** Open-Meteo historical archive, `https://archive-api.open-meteo.com/v1/archive` (free, no key).
- **Variables:** hourly temperature, wind speed, wind gusts, precipitation, sea-level pressure, one call per store location.
- **Timezone:** requested in **UTC**, then converted to `America/New_York` so it lines up with the naive local timestamps in the sales data. September is daylight time (EDT, UTC-4); the brief calls sales time "EST", so I treat it as US Eastern local time.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = next(d for d in ('data', '../data', '../../data') if os.path.isdir(d))
MISSING_TOKENS = {'', 'null', 'n/a', 'na', 'nan', 'none'}
DECISION_LOG = []

def log_decision(step, decision, rows_touched, rows_before=None, rows_after=None):
    DECISION_LOG.append({'step': step, 'decision': decision, 'rows_touched': int(rows_touched),
                         'rows_before': rows_before, 'rows_after': rows_after})


In [2]:
CATEGORY_MAP = {
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'electronics': 'Electronics',
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'health': 'Health', 'health & wellness': 'Health',
    'household': 'Household', 'household essentials': 'Household',
    'snacks': 'Snacks',
}
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
              '%d-%b-%Y %H:%M', '%m-%d-%Y %H:%M:%S']

def load_transactions():
    return pd.read_csv(os.path.join(DATA_DIR, 'walmart_transactions.csv'), dtype=str)

def standardize_missing(df):
    """Strip whitespace; turn '', NULL, N/A, NaN... into real NaN."""
    out = df.copy()
    touched = 0
    for c in out.columns:
        s = out[c].str.strip()
        newly_na = s.str.lower().isin(MISSING_TOKENS) & out[c].notna()
        touched += int(newly_na.sum())
        out[c] = s.mask(newly_na)
    log_decision('missing', 'strip whitespace; blank/NULL/N-A tokens -> NaN (cells, not rows)', touched, len(df), len(out))
    return out

def remove_duplicates(df):
    n0 = len(df)
    out = df.drop_duplicates()
    log_decision('duplicates', 'drop exact duplicate rows (keep first)', n0 - len(out), n0, len(out))
    n1 = len(out)
    out = out.drop_duplicates('transaction_id')
    log_decision('duplicates', 'drop repeated transaction_id (keep first)', n1 - len(out), n1, len(out))
    return out

def clean_unit_price(df):
    out = df.copy()
    raw = out['unit_price']
    dollar = int(raw.str.contains('$', regex=False, na=False).sum())
    out['unit_price'] = pd.to_numeric(raw.str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
    bad = int(out['unit_price'].isna().sum() - raw.isna().sum())
    log_decision('unit_price', "remove '$', cast text -> float", dollar, len(df), len(out))
    log_decision('unit_price', 'values that still failed numeric cast -> NaN', bad, len(df), len(out))
    return out

def clean_quantity(df):
    out = df.copy()
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    n0 = len(out)
    out = out[out['quantity'] > 0]
    log_decision('quantity', 'drop rows with negative/zero/unparseable quantity (entry errors; sign-flip not assumed)', n0 - len(out), n0, len(out))
    return out

def parse_timestamps(df):
    out = df.copy()
    ts = pd.Series(pd.NaT, index=out.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        todo = ts.isna() & out['timestamp'].notna()
        ts[todo] = pd.to_datetime(out.loc[todo, 'timestamp'], format=fmt, errors='coerce')
    unparsed = int((ts.isna() & out['timestamp'].notna()).sum())
    missing = int(out['timestamp'].isna().sum())
    out['timestamp'] = ts
    log_decision('timestamp', f'parse {len(TS_FORMATS)} mixed formats into one datetime (naive local time); {unparsed} unparseable', int(ts.notna().sum()), len(df), len(out))
    log_decision('timestamp', 'missing timestamps kept (NaT) but excluded from any date-based analysis; not imputed', missing, len(df), len(out))
    return out

def clean_store_id(df):
    out = df.copy()
    raw = out['store_id']
    out['store_id'] = (raw.str.upper().str.replace('_', '-', regex=False)
                          .str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True))
    log_decision('store_id', "uppercase, strip, '_'/missing '-' -> canonical 'FL-###'", int((raw != out['store_id']).sum()), len(df), len(out))
    return out

def clean_category(df):
    out = df.copy()
    raw = out['category']
    out['category'] = raw.str.lower().map(CATEGORY_MAP)
    log_decision('category', f'collapse {raw.nunique()} spellings into {out["category"].nunique()} canonical categories', int((raw != out['category']).sum()), len(df), len(out))
    return out

def clean_transactions():
    DECISION_LOG.clear()
    df = load_transactions()
    for step in (standardize_missing, remove_duplicates, clean_unit_price, clean_quantity,
                 parse_timestamps, clean_store_id, clean_category):
        df = step(df)
    df['product_name_raw'] = df['product_name']
    return df.reset_index(drop=True)


In [3]:
import time, requests

WEATHER_URL = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_VARS = ['temperature_2m', 'wind_speed_10m', 'wind_gusts_10m', 'precipitation', 'pressure_msl']

def fetch_weather(lat, lon, start, end, retries=4, backoff=2):
    """Hourly weather from Open-Meteo for one point. Retries with exponential backoff and fails loudly."""
    params = dict(latitude=lat, longitude=lon, start_date=start, end_date=end,
                  hourly=','.join(WEATHER_VARS), timezone='UTC')   # request UTC, convert ourselves
    last = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(WEATHER_URL, params=params, timeout=30)
            r.raise_for_status()
            df = pd.DataFrame(r.json()['hourly']).rename(columns={'time': 'time_utc'})
            df['time_utc'] = pd.to_datetime(df['time_utc'])
            df['time_local'] = (df['time_utc'].dt.tz_localize('UTC')
                                .dt.tz_convert('America/New_York').dt.tz_localize(None))
            df['source'] = 'open-meteo-archive'
            return df
        except (requests.RequestException, KeyError, ValueError) as e:
            last = e
            print(f'attempt {attempt}/{retries} failed: {e}')
            time.sleep(backoff ** attempt)
    raise RuntimeError(f'weather API failed after {retries} attempts: {last}')


In [4]:
stores = pd.read_csv(os.path.join(DATA_DIR, 'store_locations.csv'))
stores['store_id'] = stores['store_id'].str.upper().str.replace('_', '-', regex=False).str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True)
stores = stores.drop_duplicates('store_id')
stores.loc[stores['store_id'] == 'FL-619', ['latitude', 'longitude']] = (30.4383, -84.2807)   # Tallahassee fill, see notebook 05

# pull Aug 31 - Oct 1 UTC so every local September hour is covered after the UTC -> Eastern shift
frames = []
for r in stores.itertuples():
    w = fetch_weather(r.latitude, r.longitude, '2024-08-31', '2024-10-01')
    w.insert(0, 'store_id', r.store_id)
    frames.append(w)
weather = pd.concat(frames, ignore_index=True)
weather = weather[(weather['time_local'] >= '2024-09-01') & (weather['time_local'] < '2024-10-01')]
print('rows:', len(weather), '| stores:', weather['store_id'].nunique())
weather.head()

rows: 8640 | stores: 12


,store_id,time_utc,temperature_2m,wind_speed_10m,wind_gusts_10m,precipitation,pressure_msl,time_local,source
28,FL-239,2024-09-01 04:00:00,24.8,7.9,14.8,0.0,1015.2,2024-09-01 00:00:00,open-meteo-archive
29,FL-239,2024-09-01 05:00:00,24.4,5.5,13.0,0.0,1015.0,2024-09-01 01:00:00,open-meteo-archive
30,FL-239,2024-09-01 06:00:00,24.1,5.8,9.0,0.0,1014.5,2024-09-01 02:00:00,open-meteo-archive
31,FL-239,2024-09-01 07:00:00,23.9,5.4,10.1,0.0,1014.0,2024-09-01 03:00:00,open-meteo-archive
32,FL-239,2024-09-01 08:00:00,23.7,4.9,8.6,0.0,1013.8,2024-09-01 04:00:00,open-meteo-archive


In [5]:
print(weather[['time_utc', 'time_local']].head(3))
print('\nmissing values:'); print(weather.isna().sum())
print('\nrows per store (expect 720 = 30 days x 24h):'); print(weather.groupby('store_id').size().unique())
print('\npeak daily max wind gust by store (km/h):')
print(weather.groupby('store_id')['wind_gusts_10m'].max().sort_values(ascending=False).head(5))

              time_utc          time_local
28 2024-09-01 04:00:00 2024-09-01 00:00:00
29 2024-09-01 05:00:00 2024-09-01 01:00:00
30 2024-09-01 06:00:00 2024-09-01 02:00:00

missing values:
store_id          0
time_utc          0
temperature_2m    0
wind_speed_10m    0
wind_gusts_10m    0
precipitation     0
pressure_msl      0
time_local        0
source            0
dtype: int64

rows per store (expect 720 = 30 days x 24h):
[720]

peak daily max wind gust by store (km/h):
store_id
FL-445    124.6
FL-105    117.7
FL-078    113.8
FL-412    106.6
FL-619     97.6
Name: wind_gusts_10m, dtype: float64


In [6]:
# join check: daily weather must cover every day in the (cleaned) sales data
tx_clean = clean_transactions()
days_sales = set(tx_clean['timestamp'].dropna().dt.date)
days_weather = set(weather['time_local'].dt.date)
print('sales days:', len(days_sales), '| weather days:', len(days_weather), '| sales days missing weather:', len(days_sales - days_weather))
assert not (days_sales - days_weather)

sales days:

 30 | weather days: 30 | sales days missing weather: 0
